In [ ]:
import torch, sys,os

import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification,Trainer, TrainingArguments

sys.path.append('../')
from project.nlp_training_utils import clean_text, tokenize_function, compute_bert_metrics

## Notebook Purpose:
<b> This is notebook number 3b </b>

Gives code for training bert, or Roberta for mixtureModel

### Notebook Order
1. getData
2. downloadData
3. trainResNetModel | trainPromptTransformerClassifier | trainViTClassifier
4. localMixtureEval |  localModelEval

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
datadir = '../data/' #populate this
csv_dir = os.path.join(datadir,'justin_dataset_may_models.csv') #populat this
##For distilbert training use below
hf_model ="distilbert-base-uncased"
##For roberta training use below
# hf_model = "FacebookAI/roberta-large-mnli"
batch_size = 32
metric_name = "f1"

In [ ]:
data = pd.read_csv(csv_dir)

Clean text for use in training/inference

In [ ]:
data = data.dropna(subset = ['prompt'])

data['cleaned_prompt'] = data['prompt'].apply(lambda x : clean_text(str(x)).replace('-',' ').replace('_', ' '))


In [ ]:
data.head()

Create label2id

In [ ]:
label2id = {'PG':0, "PG13": 1, "R": 2, "X": 3, "XXX":4}
id2label = {v: k for k,v in label2id.items()}
labels = [k for k in label2id.keys()]
print(labels)

In [ ]:
# Ensure labels are integers
data['label'] = data['label'].map(label2id).astype(int)
modeling_data = data[['cleaned_prompt', 'label']]
train_df, val_df = train_test_split(modeling_data[['cleaned_prompt','label']].reset_index(drop = True), test_size=0.1, random_state=1234)


In [ ]:
# Convert Pandas DataFrames to datasets and drop the index
train_dataset = Dataset.from_pandas(
    train_df[['cleaned_prompt','label']].reset_index(drop=True))

val_dataset = Dataset.from_pandas(
    val_df[['cleaned_prompt','label']].reset_index(drop=True))

# Create DatasetDict
datasets_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(hf_model)
model = AutoModelForSequenceClassification \
    .from_pretrained(hf_model,
        num_labels = len(labels),
        id2label = id2label,
        label2id = label2id,
        ignore_mismatched_sizes=True)


In [ ]:
# Tokenize the dataset using function
encoded_dataset = datasets_dict.map(
    tokenize_function,
    batched=True,
    fn_kwargs={"column": "cleaned_prompt", "tokenizer": tokenizer}
)

In [ ]:
# Set the format to torch tensors including the 'label' column
encoded_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

In [ ]:
model_dir ='../models/'
model_type = 'roberta/'
# model_type = 'bert/'
model_name = 'promptMovieClassifier'

# Define the training arguments
output_dir = os.path.join(model_dir, model_type, model_name)
logging_dir = os.path.join(output_dir, 'logs')


In [ ]:
# Define the training arguments
training_args = TrainingArguments(
    output_dir = output_dir,
    evaluation_strategy = 'epoch',
    save_strategy = 'epoch',
    learning_rate = 2e-5,
    num_train_epochs=2,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    warmup_steps=500,
    weight_decay=0.01,
    load_best_model_at_end = True,
    metric_for_best_model = metric_name,
    logging_dir=logging_dir,
)

In [ ]:
# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['validation'],
    compute_metrics=lambda p: compute_bert_metrics(p, metric_name),  # Define your metric function
    tokenizer = tokenizer 
)

In [ ]:
trainer.train()

## Save model

In [ ]:
trainer.save_model('../models/community_seb_dataset_promptMovieClassifier_10epoch', 'promptMovieClassifier_10epoch')